In [ ]:
import os, sys
import tempfile
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.optimize as spo
from collections import defaultdict

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from alpha.engine.backtest import run_daily
from alpha.engine.strategy import Strategy
from alpha.engine.plot import plot
from alpha.engine.metrics import generate_overall_metrics, generate_monthly_returns

In [3]:
# Define the QPIStrategy class, inheriting from the Strategy base class.
class QPIStrategy(Strategy):
    max_positions = 4  # Limit the maximum number of positions in the portfolio.
    trade_id = 0  # Initialize a trade ID counter for tracking trades.
    current_trade = defaultdict(lambda: -1)  # Dictionary to store active trades.
    universe_df = None  # DataFrame to store the universe of stocks

    
    def compute_signals(self, pricing_data: pd.DataFrame) -> pd.DataFrame:
        """This method is meant to generate trading signals but is currently 
        a placeholder. It will be implemented to process price data and 
        identify trade opportunities.
        """
        return pricing_data

    def iterate(self, data: pd.DataFrame, close: pd.DataFrame, open_prices: pd.Series):
        """This method will contain the logic for executing trades based on 
        the strategy rules. It places buy and sell orders according to the strategy
        """
        positions = self.get_positions()
        long_positions = positions[positions > 0]
        long_slots = self.max_positions - len(long_positions)
        
        # Exit rules
        for symbol in long_positions.index:
            c = close[(symbol, 'Close')]
            yh = data[(symbol, 'High')].iloc[-2]
            if c > yh:
                self.order_target_value(symbol, 0, trade_id=self.current_trade[symbol])
                long_slots += 1
        
        # Entry rules
        capital_to_allocate_per_trade = self.previous_total_value / self.max_positions
        long_opportunities = self.get_opportunities(close)
        while long_slots > 0 and len(long_opportunities) > 0:
            symbol = long_opportunities.pop(0)
            if self.get_position(symbol) != 0:
                continue
        
            self.trade_id += 1
            self.current_trade[symbol] = self.trade_id
            self.order_value(symbol, capital_to_allocate_per_trade, trade_id=self.trade_id)
            long_slots -= 1

    def get_opportunities(self, close) -> list:
        # Unstack multi-index DataFrame to have symbols as index and features as columns
        df = close.unstack().dropna()
    
        # Remove benchmark symbols (e.g., $SPX) from the dataset
        df = df[~df.index.astype(str).str.startswith('$')]
    
        # Apply entry filters:
        # - QPI indicator below 15 (signal of extreme decline)
        # - Price above 200-day SMA (uptrend condition)
        # - 3-day return is negative (confirm recent decline)
        # Then sort by turnover in descending order for liquidity ranking
        df = df[
            (df['qp_indicator'] < 15) &
            (df['Close'] > df['sma_200']) &
            (df['3d_bwd_return'] < 0)
        ].sort_values('Turnover', ascending=False)
    
        # Get the list of stocks in the universe on the previous trading day
        u = self.universe_df.loc[self.previous_bar]
        u = u[u == 1].index.tolist()
    
        # Return the filtered list of opportunity tickers that are also in the universe
        return df[df.index.isin(u)].index.tolist()

In [13]:
# Define the DVOStrategy class, inheriting from the Strategy base class.
class DVO2Strategy(Strategy):
    max_positions = 10  # Limit the maximum number of positions in the portfolio.
    trade_id = 0  # Initialize a trade ID counter for tracking trades.
    current_trade = defaultdict(lambda: -1)  # Dictionary to store active trades.
    universe_df = None  # DataFrame to store the universe of stocks

    def compute_signals(self, pricing_data: pd.DataFrame) -> pd.DataFrame:
        """This method is meant to generate trading signals but is currently 
        a placeholder. It will be implemented to process price data and 
        identify trade opportunities.
        """
        return pricing_data

    def iterate(self, data: pd.DataFrame, close: pd.DataFrame, open_prices: pd.Series):
        """This method will contain the logic for executing trades based on 
        the strategy rules. It places buy and sell orders according to the strategy
        """
        positions = self.get_positions()
        long_positions = positions[positions > 0]
        long_slots = self.max_positions - len(long_positions)
        
        # Exit rules
        for symbol in long_positions.index:
            c = close[(symbol, 'Close')]
            yh = data[(symbol, 'High')].iloc[-2]
            if c > yh:
                self.order_target_value(symbol, 0, trade_id=self.current_trade[symbol])
                long_slots += 1
        
        # Entry rules
        capital_to_allocate_per_trade = self.previous_total_value / self.max_positions
        long_opportunities = self.get_opportunities(close)
        while long_slots > 0 and len(long_opportunities) > 0:
            symbol = long_opportunities.pop(0)
            if self.get_position(symbol) != 0:
                continue
        
            self.trade_id += 1
            self.current_trade[symbol] = self.trade_id
            self.order_value(symbol, capital_to_allocate_per_trade, trade_id=self.trade_id)
            long_slots -= 1

    def get_opportunities(self, close) -> list:
        # Unstack multi-index DataFrame to have symbols as index and features as columns
        df = close.unstack().dropna()
    
        # Remove benchmark symbols (e.g., $SPX) from the dataset
        df = df[~df.index.astype(str).str.startswith('$')]
    
        # Apply entry filters:
        df = df[
            (df['dv2'] < 10) &
            (df['Close'] > df['sma_200']) &
            (df['p126d_return'] > 0)
        ].sort_values('natr', ascending=False)
    
        # Get the list of stocks in the universe on the previous trading day
        u = self.universe_df.loc[self.previous_bar]
        u = u[u == 1].index.tolist()
    
        # Return the filtered list of opportunity tickers that are also in the universe
        return df[df.index.isin(u)].index.tolist()

In [5]:
def optimize_portfolio(prices, silent=False, target='sharpe'):
    # this is the function to be optimized
    def neg_sharpe_ratio(x):
        daily_returns = prices.pct_change().fillna(0)
        portfolio_daily_returns = (daily_returns * x).sum(axis=1)
        sdd_ret = portfolio_daily_returns.std()
        return -1 * portfolio_daily_returns.mean() / sdd_ret * (252 ** .5)

    def neg_sortino_ratio(x):
        daily_returns = prices.pct_change().fillna(0)
        portfolio_daily_returns = (daily_returns * x).sum(axis=1)
        sdd_ret = portfolio_daily_returns[portfolio_daily_returns < 1].std()
        return -1 * portfolio_daily_returns.mean() / sdd_ret * (252 ** .5)

    def neg_calmar_ratio(x):
        daily_returns = prices.pct_change().fillna(0)
        portfolio_daily_returns = (daily_returns * x).sum(axis=1)
        portfolio_total = (portfolio_daily_returns + 1).cumprod()
        portfolio_annual = portfolio_total.iloc[-1] ** (252 / len(prices)) - 1
        cummax = portfolio_total.cummax()
        drawdown = (portfolio_total - cummax) / cummax
        return -portfolio_annual / abs(drawdown.min())

    # this is the function that constrains the sum of allocation to be maximum 100%
    def constraint_total(x):
        return x.sum() - 1

    # this is the tuple that represents the boundaries of each allocation between 0 and 100%
    bounds = tuple([(0.0, 1.0)] * len(prices.columns))

    # first guess allocation: evenly distributed
    alloc_guess = np.full((len(prices.columns),), 1.0 / len(prices.columns), dtype=float)

    if target == 'sharpe':
        fn_optimize = neg_sharpe_ratio
    elif target == 'sortino':
        fn_optimize = neg_sortino_ratio
    elif target == 'calmar':
        fn_optimize = neg_calmar_ratio
    else:
        raise ValueError('Invalid target')

    # find the allocations for the optimal portfolio
    allocs = spo.minimize(fn_optimize, alloc_guess, method='SLSQP', options={'disp': not silent},
                          constraints=[{'type': 'eq', 'fun': constraint_total}],
                          bounds=bounds).x

    # get daily portfolio value
    normalized_prices = prices / prices.iloc[0]
    allocations = normalized_prices * allocs
    port_val = allocations.sum(axis=1)
    drs = port_val.div(port_val.shift(1))[1:]  # daily returns

    cr = port_val.iloc[-1] - 1
    adr = drs.mean() - 1
    sddr = drs.std()
    sr = adr / sddr * (252 ** .5)
    return allocs

In [ ]:
tmp = Path(tempfile.gettempdir())
qpi = QPIStrategy.read_pickle(tmp / 'QPIStrategy.pkl')
dvo = DVO2Strategy.read_pickle(tmp / 'DVO2Strategy.pkl')

prices = pd.concat([qpi.results['total_value'], dvo.results['total_value']], axis=1)
prices

prices.columns = ['QPIStrategy', 'DVO2Strategy']

display(optimize_portfolio(prices.loc[prices.index <= '2012-12-31']))

In [18]:
df = pd.DataFrame()
df['daily_rets_qpi'] = qpi.results['daily_returns']
df['daily_rets_dvo'] = dvo.results['daily_returns']
df['benchmark'] = qpi.results['$SPX']

df = df.loc[df.index >= '2013-01-01']

allocs = np.array([0.30, 0.70])
daily_rets = (df[['daily_rets_qpi', 'daily_rets_dvo']] * allocs).sum(axis=1)
cumulative_rets = (1 + daily_rets).cumprod()
cumulative_rets = cumulative_rets / cumulative_rets.iloc[0]
benchmark = df['benchmark'] / df['benchmark'].iloc[0]

In [ ]:
def get_drawdown(total_value):
    running_max = total_value.cummax()
    return (total_value - running_max) / running_max


tmp = Path(tempfile.gettempdir())
plot(
    strategy_total_value=cumulative_rets,
    strategy_drawdown=get_drawdown(cumulative_rets),
    benchmark_total_value=benchmark,
    benchmark_drawdown=get_drawdown(benchmark),
    use_log_scale=True,
    save_to=tmp / 'portfolio.png'
)

In [20]:
summary = pd.DataFrame()
summary['Strategy'] = generate_overall_metrics(
    total_value=cumulative_rets,
    portfolio_value=cumulative_rets,
    series_to_correlate=cumulative_rets.pct_change().fillna(0)
)
summary['Benchmark'] = generate_overall_metrics(
    total_value=benchmark,
    series_to_correlate=cumulative_rets.pct_change().fillna(0)
)
summary

,Strategy,Benchmark
Start,2013-01-02 00:00:00,2013-01-02 00:00:00
End,2025-05-30 00:00:00,2025-05-30 00:00:00
Duration [days],3122,3122
Exposure Time [%],100,100
Start [$],1.0,1.0
Final [$],9.557558,4.042402
Peak [$],9.611429,4.201358
Return [%],855.755772,304.240215
Return (Ann.) [%],19.986157,11.935133
Volatility (Ann.) [%],18.095113,17.278382


In [21]:
mr = generate_monthly_returns(cumulative_rets, add_sharpe_ratios=True, add_max_drawdowns=True)
for c in mr.columns[:-1]:
    mr[c] = mr[c].apply(lambda x: f'{x:.1%}' if not pd.isna(x) else "")
mr['Sharpe Ratio'] = mr['Sharpe Ratio'].apply(lambda x: f'{x:.2f}' if not pd.isna(x) else "")
mr

month,1,2,3,4,5,6,7,8,9,10,11,12,Annual Return,Max Drawdown,Sharpe Ratio
year,,,,,,,,,,,,,,,
2025,7.4%,-1.7%,-1.4%,0.9%,3.0%,,,,,,,,8.2%,-14.1%,0.85
2024,1.1%,4.4%,-0.4%,0.0%,5.5%,3.6%,2.6%,-2.0%,2.9%,-3.3%,8.8%,-5.8%,17.8%,-12.8%,1.12
2023,-0.9%,4.0%,-2.7%,3.9%,-3.2%,8.6%,8.1%,-1.1%,-3.4%,-5.9%,6.1%,4.7%,18.3%,-12.7%,1.25
2022,-9.4%,2.5%,4.1%,-7.6%,3.7%,-3.5%,3.9%,1.1%,-4.0%,10.2%,6.6%,-0.7%,5.0%,-17.5%,0.34
2021,-1.3%,7.6%,8.6%,2.8%,-2.0%,-1.2%,-1.6%,-0.4%,-3.6%,5.2%,-4.3%,6.6%,16.5%,-10.9%,0.89
2020,-2.7%,-4.0%,-7.5%,28.4%,5.7%,1.4%,4.4%,2.4%,0.7%,-0.6%,29.9%,4.7%,72.7%,-26.8%,1.91
2019,1.9%,2.3%,1.3%,0.6%,-8.7%,7.7%,-0.2%,-0.8%,2.2%,3.1%,5.7%,3.6%,19.3%,-9.2%,1.36
2018,9.0%,-2.9%,-1.8%,1.2%,7.5%,-0.6%,1.9%,1.9%,3.4%,-6.9%,2.2%,-5.8%,8.1%,-15.1%,0.51
2017,3.7%,2.8%,-2.9%,0.8%,0.2%,-1.9%,2.2%,-1.9%,1.9%,0.6%,4.4%,3.6%,13.9%,-7.1%,1.14


# Fama-French 3-Factor Model

In [ ]:
def get_ff_factor_model(factors=3, output_dir=None, silent=True):
    if output_dir is None:
        output_dir = tempfile.gettempdir()

    if factors == 3:
        url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip"
        csv_filename = "F-F_Research_Data_Factors_daily.CSV"
        zip_filename = "F-F_Research_Data_Factors_daily_CSV.zip"
        skiprows = 4
    elif factors == 5:
        url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
        csv_filename = "F-F_Research_Data_5_Factors_2x3_daily.CSV"
        zip_filename = "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
        skiprows = 2
    else:
        raise ValueError("Factors must be either 3 or 5.")

    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Download and unzip the file
    download_and_unzip(url, output_dir, zip_filename, silent=silent)
    if not silent:
        print("Operation completed successfully.")

    # Read the CSV file
    csv_path = os.path.join(output_dir, csv_filename)
    ff = pd.read_csv(csv_path, skiprows=skiprows, index_col=0).iloc[:-1]
    ff.index = pd.to_datetime(ff.index, format='%Y%m%d')
    return ff

In [25]:
import os
import requests
import zipfile
from datetime import datetime, timedelta

def is_older_than_24h(file_path):
    path = Path(file_path)
    # Check if file exists
    if not path.exists():
        return True

    # Get last modification time
    mtime = datetime.fromtimestamp(path.stat().st_mtime)
    # Get current time
    now = datetime.now()
    # Calculate time difference
    time_diff = now - mtime
    return time_diff > timedelta(hours=24)
    
    
def download_and_unzip(url, output_dir, zip_filename, silent=True):
    # Define paths
    zip_path = os.path.join(output_dir, zip_filename)

    if not is_older_than_24h(zip_path):
        if not silent:
            print(f"File {zip_path} is up to date.")
        return

    # Download the ZIP file
    if not silent:
        print(f"Downloading file from {url}...")
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        if not silent:
            print("Download successful.")
    else:
        raise Exception(f"Failed to download file. HTTP Status Code: {response.status_code}")

    # Save the ZIP file
    if not silent:
        print(f"Saving ZIP file to {zip_path}...")
    with open(zip_path, "wb") as file:
        file.write(response.content)

    # Unzip the contents
    if not silent:
        print("Unzipping the file...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
        if not silent:
            print(f"Files extracted to {output_dir}")

In [27]:
import statsmodels.api as sm

def get_alpha_and_betas(daily_returns, factors=3, yearly_alpha=True, return_model=False, min_periods=42):
    ff = get_ff_factor_model(factors=factors)
    strategy = daily_returns * 100
    strategy.name = 'strategy'

    ff = ff.join(strategy, how='inner').dropna()
    if len(ff) < min_periods:
        return None

    ff['excess_return'] = ff['strategy'] - ff['RF']

    Y = ff['excess_return']
    if factors == 3:
        X = ff[['Mkt-RF', 'SMB', 'HML']]
    elif factors == 5:
        X = ff[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]
    else:
        raise ValueError("Factors must be either 3 or 5.")

    # Add a constant to the independent variables (for the intercept, which is alpha)
    X = sm.add_constant(X).rename(columns={"const": "alpha"})

    # Run the OLS regression
    model = sm.OLS(Y, X).fit()

    results = model.params
    if yearly_alpha:
        results['alpha'] = (1 + results['alpha']/100) ** 252 - 1

    if not return_model:
        return results
    return results, model

In [28]:
results, model = get_alpha_and_betas(daily_rets, return_model=True)
print(results)
print(model.summary())

alpha     0.087616
Mkt-RF    0.792499
SMB       0.069297
HML       0.065093
dtype: float64
                            OLS Regression Results                            
Dep. Variable:          excess_return   R-squared:                       0.602
Model:                            OLS   Adj. R-squared:                  0.602
Method:                 Least Squares   F-statistic:                     1554.
Date:                Sat, 31 May 2025   Prob (F-statistic):               0.00
Time:                        23:56:48   Log-Likelihood:                -3309.8
No. Observations:                3080   AIC:                             6628.
Df Residuals:                    3076   BIC:                             6652.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------